# Validating (and Repairing) an Existing Model with `pyshifty`

This is the [`pyshifty`](https://pypi.org/project/pyshifty/) version of the
**"Validating and Assessing an Existing Model"** demo. It loads the *same*
ontologies and the *same* Medium Office model, but validates with the `pyshifty`
SHACL engine by default.

With the `pyshifty` engine, `model.validate(...)` returns an
**`AlgebraicValidationContext`** instead of the legacy `ValidationContext`.
Rather than re-parsing a flattened W3C SHACL report, it consumes pyshifty's
*algebraic* output directly and can drive pyshifty's **symbolic repair** engine:
each failing focus node gets a **repair tree** of typed *holes*, and every
candidate fix is run through a **soundness gate** (it must not introduce a new
violation).

This notebook has two parts:

1. **Faithful port** — load the same libraries + Medium Office model and validate
   it with `pyshifty`, then read the *violation horizon* (witnesses + reasons).
2. **A repair playground** — a small, legible model built on the *same* Brick
   ontology, with several sections where you can **experiment with different ways
   of fixing the graph** (synthesis, template reuse, template mint, lifting to
   BuildingMOTIF templates, deletion-direction repair, …).

## Part 1 — Faithful port: validate the Medium Office model

### A. Imports

In [ ]:
import os
from rdflib import Namespace, Graph, Literal

from buildingmotif import BuildingMOTIF
from buildingmotif.dataclasses import Model, Library, AlgebraicValidationContext
from buildingmotif.namespaces import BMOTIF, BRICK, SH, RDFS, PARAM, A

### B. Create a BuildingMOTIF instance

`pyshifty` is the default engine, so validation returns the algebraic validation + repair report without extra configuration.

In [ ]:
building_motif = BuildingMOTIF("sqlite://")

### C. Load the same BuildingMOTIF libraries

These are exactly the libraries the original validation notebook loads. Loading
QUDT + Brick + Guideline 36 may take time on the first run while OntoEnv resolves
and caches their imports. Treat unresolved-import warnings as incomplete requirements,
not as a successful load.

- **QUDT** — units and quantity kinds
- **Brick** — the Brick ontology + shapes
- **ASHRAE Guideline 36** — section shapes *and* templates for G36-compliant equipment
- **constraints** — the builtin constraint vocabulary

In [ ]:
qudtqk = Library.from_ontology("http://qudt.org/2.1/vocab/quantitykind")
qudtunit = Library.from_ontology("http://qudt.org/2.1/vocab/unit")
brick = Library.from_ontology("../libraries/brick/Brick.ttl")
ashrae_g36 = Library.from_directory("../libraries/ashrae/guideline36/")
constraints = Library.from_ontology("constraints/constraints.ttl")  # builtin library

### D. Load the Medium Office model

The same compiled Brick model as the original demo.

In [ ]:
BLDG = Namespace("http://example.org/building/")
medium_office_model = Model.from_file(
    os.path.join("mediumOffice-validation", "mediumOffice_brick_compiled.ttl")
)
print(medium_office_model.graph.serialize(format="turtle")[:600], "...")

### E. Validate with `pyshifty`

The project manifest (`mediumOffice_constraints.ttl`) uses a few non-standard
constraint components (`constraint:exactCount`, `sh:qualifiedValueShape`). The
`pyshifty` engine works on plain SHACL, so here we express **one real requirement**
from that manifest as plain SHACL: *every `brick:AHU` must carry a
`brick:Supply_Air_Temperature_Setpoint`.*

The Medium Office AHUs are richly instrumented with **sensors**, but they have no
supply-air-temperature **setpoint** — so this is a genuine gap in the real model.

In [ ]:
ahu_manifest = Graph().parse(data='''
@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl:   <http://www.w3.org/2002/07/owl#> .
@prefix :      <urn:medium-office/manifest#> .

: a owl:Ontology .

:ahu-supply-air-temp-setpoint a sh:NodeShape ;
    sh:targetClass brick:AHU ;
    sh:property [
        sh:path brick:hasPoint ;
        sh:minCount 1 ;
        sh:class brick:Supply_Air_Temperature_Setpoint ;
    ] .
''', format="turtle")
ahu_manifest_lib = Library.from_ontology(ahu_manifest)

ctx = medium_office_model.validate(
    [ahu_manifest_lib.get_shape_collection()],
    error_on_missing_imports=False,
)

print("Report type :", type(ctx).__name__)
print("Model valid :", ctx.valid)

### F. Read the reasons for failure (the *violation horizon*)

The algebraic report exposes one **`RepairWitness`** per failing
`(focus node, statement)`. For drop-in compatibility with the legacy report it
also offers `diffset` (a dict of focus node → set of failures) — but here each
failure is a `RepairWitness` carrying a structured `reason()` and a `repair_tree`.

In [ ]:
print(ctx.report_string)
print("=" * 70)
for focus, witnesses in ctx.diffset.items():
    print(focus)
    for w in witnesses:
        print("  - " + w.reason())

### G. Auto-generate fixes with `as_templates()`

Just like the legacy `ValidationContext`, the algebraic context can lift the best
sound repair per failure into BuildingMOTIF **templates** — but every repair here
has been **soundness-gated** first. (On a dense, fully-instrumented model the
reuse-first candidate search can produce large templates; the *playground* in
Part 2 is where we explore the individual proposals in detail.)

In [ ]:
generated_templates = ctx.as_templates()
print(f"generated {len(generated_templates)} repair template(s)")
for t in generated_templates[:1]:
    print("-" * 70)
    print(t.body.serialize())
    print("parameters:", t.parameters)

## Part 2 — A repair playground

The Medium Office model is large and densely instrumented, which makes the
*individual* repair proposals hard to read. To actually **experiment with
different ways of fixing a graph**, we now build a small, legible model on the
**same Brick ontology**.

The scenario:

- `bldg:ahu1` is a `brick:AHU` with **no points** → it violates our shape.
- `bldg:loose_sat` is a stray `brick:Supply_Air_Temperature_Sensor` sitting in
  the graph, unattached → a candidate the engine can **reuse**.
- The shape requires every AHU to have at least one
  `brick:Supply_Air_Temperature_Sensor`.

We also hand the engine a small **repair-template library** so we can compare
*synthesis*, *template reuse*, and *template mint* side by side.

In [ ]:
# A small model on the same Brick ontology
play_model = Model.create(Namespace("urn:play/"))
play_model.add_triples((BLDG["ahu1"], A, BRICK.AHU))
play_model.add_triples((BLDG["loose_sat"], A, BRICK.Supply_Air_Temperature_Sensor))

# The requirement, as plain SHACL
play_shapes = Graph().parse(data='''
@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl:   <http://www.w3.org/2002/07/owl#> .
@prefix :      <urn:play/manifest#> .

: a owl:Ontology .

:ahu-sat a sh:NodeShape ;
    sh:targetClass brick:AHU ;
    sh:property [
        sh:path brick:hasPoint ;
        sh:minCount 1 ;
        sh:class brick:Supply_Air_Temperature_Sensor ;
    ] .
''', format="turtle")
play_shapes_lib = Library.from_ontology(play_shapes)

# A couple of repair templates (the engine's domain vocabulary)
repair_lib = Library.create("playground-repair-templates")

_b = Graph(); _b.add((PARAM["name"], A, BRICK.Supply_Air_Temperature_Sensor))
repair_lib.create_template("make-supply-air-temperature-sensor", _b)

_b = Graph(); _b.add((PARAM["name"], A, BRICK.Temperature_Sensor))
repair_lib.create_template("make-temperature-sensor", _b)

print(play_model.graph.serialize())

### Validate the playground

Passing `repair_libraries=[repair_lib]` lets the engine use our templates as a
candidate generator (on top of recursive synthesis and pyshifty's native guesses).

In [ ]:
def validate_playground():
    # (Re)validate the playground model — handy to re-run after edits.
    return play_model.validate(
        [play_shapes_lib.get_shape_collection()],
        repair_libraries=[repair_lib],
        error_on_missing_imports=False,
    )

pctx = validate_playground()
print("valid     :", pctx.valid)
print("witnesses :", len(pctx.witnesses))

w = pctx.witnesses[0]
print("\nfocus  :", w.focus)
print("reason :", w.reason())
print("blocked:", w.is_blocked)
print("\nrepair tree:")
print(w.repair_tree.explain())

---
## Experiment A — Browse the ranked, gated proposals

`witness.proposals()` returns ranked `RepairProposal`s. Each one's `ΔG` has been
run through pyshifty's gate:

- `is_sound` — proved to introduce **no new** violation
- `is_progress` — proved to **remove** the violation
- `origin` — `synthesized` (recursive), `template:…`, or `pyshifty-candidate`
- `reused_nodes` — existing model nodes reused instead of minted

Notice the top proposal **reuses `loose_sat`** (synthesis is reuse-first), while
the template proposal **mints** a fresh, correctly-typed sensor. The
`pyshifty-candidate` rows are flat guesses that are sound but make no progress
(binding the point to an arbitrary node does not satisfy `sh:class`).

In [ ]:
w = pctx.witnesses[0]
for p in w.proposals():
    reused = sorted(str(n).split("/")[-1] for n in p.reused_nodes)
    print(f"[{p.origin:34s}] sound={p.is_sound!s:5} progress={p.is_progress!s:5} "
          f"additions={p.num_additions} reused={reused}")
    for (s, pred, o) in p.additions:
        print(f"      + {s} {pred} {o}")
    for (s, pred, o) in p.deletions:
        print(f"      - {s} {pred} {o}")

---
## Experiment B — Apply the best repair and re-validate

A proposal can be applied to the pyshifty session to produce a patched graph.
`RepairOutcome` records what the fix does, and `advance()` returns a fresh session
over `G ⊕ ΔG` so we can confirm the violation is gone with nothing new introduced.

> Try changing `proposals()[0]` to `[1]`, `[2]`, … to apply a different fix.

In [ ]:
best = pctx.witnesses[0].proposals()[0]
print("chosen proposal :", best.origin)
print("  fixed         :", len(best.outcome.fixed))
print("  introduced    :", len(best.outcome.introduced))

patched_session = best.advance(pctx.session)
print("  remaining violations after applying:", len(patched_session.witnesses()))

---
## Experiment C — Reuse vs. mint: pick a specific alternative

Every gated proposal is a valid fix; you may prefer a particular one. Here we
pull out the proposal that **mints** from a named template versus the one that
**reuses** an existing node, and lift whichever we like into a BuildingMOTIF
template with `RepairProposal.as_template()`.

In [ ]:
proposals = pctx.witnesses[0].proposals()

reuse_p = next((p for p in proposals if p.reused_nodes), None)
mint_p  = next((p for p in proposals if p.origin.startswith("template:")), None)

if reuse_p:
    print("REUSE proposal:", reuse_p.origin, "-> reuses",
          sorted(str(n).split("/")[-1] for n in reuse_p.reused_nodes))
    for (s, pred, o) in reuse_p.additions:
        print("   +", s, pred, o)

if mint_p:
    print("\nMINT proposal:", mint_p.origin)
    print(mint_p.as_template().body.serialize())

---
## Experiment D — Opinionated auto-fix with `as_templates()`

`as_templates()` keeps the single best sound repair per failure and merges them,
exposing each freshly-minted individual as a template parameter. We fill the
parameters, add the result to the model, and re-validate — the model now conforms.

In [ ]:
generated = pctx.as_templates()
for t in generated:
    print("-" * 70)
    print(t.body.serialize())
    bindings = {param: BLDG[f"new_{param}"] for param in t.parameters}
    play_model.add_graph(t.substitute(bindings).to_graph())

after = validate_playground()
print("Model valid after repair:", after.valid)
print(play_model.graph.serialize())

---
## Experiment E — The full menu: `all_repair_templates()`

Where `as_templates()` is opinionated (one merged best repair per failure),
`all_repair_templates()` returns **every** sound, gated repair as a separate
template, grouped by focus — the full set of alternatives to choose from.
`RepairWitness.repair_templates()` does the same for a single failure.

> We re-create a fresh, still-broken playground here so there is something to
> repair (Experiment D already fixed `ahu1`).

In [ ]:
fresh_model = Model.create(Namespace("urn:play2/"))
fresh_model.add_triples((BLDG["ahu9"], A, BRICK.AHU))
fresh_model.add_triples((BLDG["loose_sat2"], A, BRICK.Supply_Air_Temperature_Sensor))

fctx = fresh_model.validate(
    [play_shapes_lib.get_shape_collection()],
    repair_libraries=[repair_lib],
    error_on_missing_imports=False,
)

for focus, templates in fctx.all_repair_templates().items():
    print(f"{focus}: {len(templates)} alternative repair template(s)")
    for t in templates:
        objs = ", ".join(sorted(str(o).split('/')[-1].split('#')[-1] for o in t.body.objects()))
        print("  -", t.name, "->", objs)

---
## Experiment F — Recursive synthesis without any templates

When a hole must conform to a sub-shape (`sh:node`/`sh:class`), the engine can
build a structurally-correct value on its own — minting a node and recursively
repairing it against the sub-shape — **with no templates at all**. It composes to
arbitrary depth (bounded by a build-fuel budget): here a node must reach, via a
chain of properties, a value of a given class, and the engine materializes the
**whole chain** in one sound repair.

In [ ]:
EX = Namespace("http://ex/")

deep_shapes = Graph().parse(data='''
@prefix sh:  <http://www.w3.org/ns/shacl#> .
@prefix ex:  <http://ex/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix :    <urn:deep/shapes#> .

: a owl:Ontology .

ex:Sub a sh:NodeShape ;
    sh:property [ sh:path ex:q ; sh:minCount 1 ; sh:class ex:Widget ] .

ex:Root-shape a sh:NodeShape ;
    sh:targetClass ex:Root ;
    sh:property [ sh:path ex:p ; sh:minCount 1 ; sh:node ex:Sub ] .
''', format="turtle")
deep_lib = Library.from_ontology(deep_shapes)

deep_model = Model.create(Namespace("urn:deep/"))
deep_model.add_triples((EX["root1"], A, EX.Root))

deep_ctx = deep_model.validate(
    [deep_lib.get_shape_collection()], error_on_missing_imports=False
)
fix = deep_ctx.witnesses[0].proposals()[0]
print("origin:", fix.origin, "| sound:", fix.is_sound, "| progress:", fix.is_progress)
print("synthesized chain:")
for (s, p, o) in fix.additions:
    print("   +", s, p, o)

---
## Experiment G (bonus) — Deletion-direction repair (`sh:not`)

Not every fix is an addition. A violated `sh:not` constraint can only be repaired
by **deletion** — and pyshifty's repair calculus handles that direction too, with
the gate validating the deletion exactly the same way.

In [ ]:
not_shapes = Graph().parse(data='''
@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl:   <http://www.w3.org/2002/07/owl#> .
@prefix :      <urn:not/shapes#> .

: a owl:Ontology .

# An AHU must NOT be marked decommissioned
:no-decommissioned a sh:NodeShape ;
    sh:targetClass brick:AHU ;
    sh:not [ sh:path brick:status ; sh:hasValue "decommissioned" ] .
''', format="turtle")
not_lib = Library.from_ontology(not_shapes)

not_model = Model.create(Namespace("urn:not/"))
not_model.add_triples((BLDG["ahu_bad"], A, BRICK.AHU))
not_model.add_triples((BLDG["ahu_bad"], BRICK.status, Literal("decommissioned")))

not_ctx = not_model.validate(
    [not_lib.get_shape_collection()], error_on_missing_imports=False
)
fix = not_ctx.witnesses[0].proposals()[0]
print("sound:", fix.is_sound, "| progress:", fix.is_progress)
print("deletions proposed:")
for (s, p, o) in fix.deletions:
    print(f"   - {s} {p} {o}")
print("additions proposed:", fix.num_additions)

---
## Your turn — scratchpad

The playground is built to be poked at. Some things to try:

- **Change the requirement.** Edit `play_shapes` to require a different class
  (e.g. `brick:Mixed_Air_Temperature_Sensor`) or a higher `sh:minCount`, then
  call `validate_playground()` again.
- **Add reuse candidates.** Add more stray sensors to `play_model` and watch the
  `reused_nodes` on the proposals change.
- **Add a repair template** to `repair_lib` and see a new `template:…` proposal
  appear.
- **Apply a non-default proposal** with `.advance(pctx.session)` and inspect the
  resulting `witnesses()`.

> Re-running cells that call `Library.create(...)` with a name that already exists
> will error — restart the kernel if you hit that.

In [ ]:
# scratchpad — edit and re-run
my_ctx = validate_playground()
for wt in my_ctx.witnesses:
    print(wt.focus, "::", wt.reason())
    for p in wt.proposals()[:3]:
        print(f"   [{p.origin}] sound={p.is_sound} progress={p.is_progress}")